# 03 — Symptom-Level Analysis

Statistical testing of inflammatory marker associations with depression, including the core novel analysis: which markers predict which specific PHQ-9 symptoms.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
PROCESSED = Path('../data/processed')
FIGURES   = Path('../figures')
FIGURES.mkdir(exist_ok=True)

df = pd.read_parquet(PROCESSED / 'nhanes_analysis_ready.parquet', engine='fastparquet')

PHQ_ITEMS = [f'DPQ0{i}0' for i in range(1, 10)]
ITEM_LABELS = {
    'DPQ010': 'Anhedonia',    'DPQ020': 'Depressed mood', 'DPQ030': 'Sleep',
    'DPQ040': 'Fatigue',      'DPQ050': 'Appetite',        'DPQ060': 'Self-worth',
    'DPQ070': 'Concentration','DPQ080': 'Psychomotor',     'DPQ090': 'Suicidality',
}
MARKERS = ['log_CRP', 'log_WBC', 'LBXNEPCT', 'LBXLYPCT',
           'log_NLR', 'LBXSAL', 'log_ferritin']
MARKER_LABELS = {
    'log_CRP':      'CRP (log)',
    'log_WBC':      'WBC (log)',
    'LBXNEPCT':     'Neutrophil %',
    'LBXLYPCT':     'Lymphocyte %',
    'log_NLR':      'NLR (log)',
    'LBXSAL':       'Albumin',
    'log_ferritin': 'Ferritin (log)',
}

# Race dummies: NH White (RIDRETH3==3) as reference
# Convert to int first so column names are race_1, race_3 etc. (not race_1.0 which breaks patsy)
df['RIDRETH3'] = df['RIDRETH3'].astype(int)
df = pd.get_dummies(df, columns=['RIDRETH3'], prefix='race', drop_first=False)
RACE_DUMMIES = [c for c in df.columns if c.startswith('race_') and not c.endswith('_3')]
COVARIATES = ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'SMQ040_bin', 'ALQ130'] + RACE_DUMMIES

print(f'N = {len(df):,}')
print(f'Depression prevalence: {df["depression"].mean()*100:.1f}%')

## 1. Binarise PHQ-9 Items

Items scored >= 2 = symptom present 'more than half the days' (clinically meaningful threshold).

In [ ]:
ITEM_BIN_COLS = []
for item in PHQ_ITEMS:
    col = f'{item}_bin'
    df[col] = (df[item] >= 2).astype(float)
    ITEM_BIN_COLS.append(col)

prev = df[ITEM_BIN_COLS].mean() * 100
prev.index = [ITEM_LABELS[c.replace('_bin', '')] for c in prev.index]
print('Symptom prevalence (item score >= 2):')
print(prev.round(1).to_string())

## 2. Weighted Logistic Regression: Binary Depression

Unadjusted and adjusted (age, sex, BMI, smoking, alcohol, race) models per marker.

In [ ]:
def run_logistic(marker, outcome, df, covariates, weight_col='WTPOOL'):
    results = {}
    for model_type, extra in [('unadjusted', []), ('adjusted', covariates)]:
        formula = f'{outcome} ~ {marker}' + ((' + ' + ' + '.join(extra)) if extra else '')
        try:
            mod = smf.glm(formula=formula, data=df,
                          family=sm.families.Binomial(),
                          freq_weights=df[weight_col]).fit(disp=0)
            results[model_type] = {
                'OR':       np.exp(mod.params[marker]),
                'CI_lower': np.exp(mod.conf_int().loc[marker, 0]),
                'CI_upper': np.exp(mod.conf_int().loc[marker, 1]),
                'pvalue':   mod.pvalues[marker],
            }
        except Exception:
            results[model_type] = {'OR': np.nan, 'CI_lower': np.nan,
                                    'CI_upper': np.nan, 'pvalue': np.nan}
    return results


logit_rows = []
for marker in MARKERS:
    res = run_logistic(marker, 'depression', df, COVARIATES)
    for model_type, vals in res.items():
        logit_rows.append({'marker': marker, 'model': model_type, **vals})

logit_df = pd.DataFrame(logit_rows)
adj_mask = logit_df['model'] == 'adjusted'
_, pvals_fdr, _, _ = multipletests(logit_df.loc[adj_mask, 'pvalue'].fillna(1), method='fdr_bh')
logit_df.loc[adj_mask, 'pvalue_fdr'] = pvals_fdr
logit_df['marker_label'] = logit_df['marker'].map(MARKER_LABELS)

print('Adjusted logistic regression results (outcome: depression PHQ-9 >= 10):')
print(logit_df[adj_mask][['marker_label','OR','CI_lower','CI_upper','pvalue','pvalue_fdr']]
      .round(3).to_string(index=False))

## 3. Forest Plot: Adjusted Odds Ratios

In [ ]:
adj = logit_df[logit_df['model'] == 'adjusted'].copy().sort_values('OR').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
y_pos = range(len(adj))
ax.errorbar(
    x=adj['OR'], y=list(y_pos),
    xerr=[adj['OR'] - adj['CI_lower'], adj['CI_upper'] - adj['OR']],
    fmt='o', color='steelblue', capsize=4, elinewidth=1.5, markersize=6
)
ax.axvline(x=1, color='crimson', linestyle='--', linewidth=1)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(adj['marker_label'].tolist())
ax.set_xlabel('Odds Ratio (95% CI)')
ax.set_title('Inflammatory Markers and Depression (PHQ-9 >= 10)\n'
             'Adjusted for age, sex, BMI, smoking, alcohol, race')

for i, (_, row) in enumerate(adj.iterrows()):
    if pd.notna(row.get('pvalue_fdr')) and row['pvalue_fdr'] < 0.05:
        ax.text(row['CI_upper'] + 0.02, i, '*',
                fontsize=14, va='center', color='darkred', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES / 'forest_plot_depression.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Symptom-Level Analysis (Core Novel Contribution)

8 markers x 9 PHQ-9 symptoms = 72 weighted logistic regressions.  
FDR correction (Benjamini-Hochberg) applied across all 72 tests jointly.

In [ ]:
symptom_rows = []
for marker in MARKERS:
    for item in PHQ_ITEMS:
        outcome = f'{item}_bin'
        formula = f'{outcome} ~ {marker} + ' + ' + '.join(COVARIATES)
        try:
            mod = smf.glm(formula=formula, data=df,
                          family=sm.families.Binomial(),
                          freq_weights=df['WTPOOL']).fit(disp=0)
            symptom_rows.append({
                'marker':   marker, 'symptom': item,
                'OR':       np.exp(mod.params[marker]),
                'CI_lower': np.exp(mod.conf_int().loc[marker, 0]),
                'CI_upper': np.exp(mod.conf_int().loc[marker, 1]),
                'pvalue':   mod.pvalues[marker],
            })
        except Exception:
            symptom_rows.append({'marker': marker, 'symptom': item,
                                  'OR': np.nan, 'CI_lower': np.nan,
                                  'CI_upper': np.nan, 'pvalue': np.nan})

symp_df = pd.DataFrame(symptom_rows)
_, symp_df['pvalue_fdr'], _, _ = multipletests(symp_df['pvalue'].fillna(1), method='fdr_bh')

print(f'Models run: {len(symp_df)}')
print(f'FDR-significant (p_fdr < 0.05): {(symp_df["pvalue_fdr"] < 0.05).sum()}')

## 5. Symptom-Marker Heatmap

In [ ]:
or_matrix  = symp_df.pivot(index='marker', columns='symptom', values='OR')
sig_matrix = symp_df.pivot(index='marker', columns='symptom', values='pvalue_fdr') < 0.05

or_matrix.index   = [MARKER_LABELS[m] for m in or_matrix.index]
or_matrix.columns = [ITEM_LABELS[c]   for c in or_matrix.columns]
sig_matrix.index   = or_matrix.index
sig_matrix.columns = or_matrix.columns

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    np.log(or_matrix),
    annot=or_matrix.round(2), fmt='',
    cmap='RdBu_r', center=0,
    linewidths=0.5,
    xticklabels=or_matrix.columns.tolist(),
    yticklabels=or_matrix.index.tolist(),
    ax=ax
)

for i, marker in enumerate(or_matrix.index):
    for j, symptom in enumerate(or_matrix.columns):
        if sig_matrix.loc[marker, symptom]:
            ax.text(j + 0.5, i + 0.5, '*',
                    ha='center', va='center', fontsize=14,
                    color='black', fontweight='bold')

ax.set_title('Odds Ratios: Inflammatory Markers x PHQ-9 Symptoms\n'
             '(log-OR colour scale; annotated with raw OR; * = FDR p < 0.05)', fontsize=11)
ax.set_xlabel('PHQ-9 Symptom')
ax.set_ylabel('Inflammatory Marker')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES / 'symptom_marker_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

symp_df['marker_label']  = symp_df['marker'].map(MARKER_LABELS)
symp_df['symptom_label'] = symp_df['symptom'].map(ITEM_LABELS)
symp_df.to_csv(PROCESSED / 'symptom_marker_results.csv', index=False)
print('Results saved.')

## 6. Sex x CRP Interaction

In [ ]:
formula_int = ('depression ~ log_CRP * RIAGENDR + RIDAGEYR + BMXBMI + '
               'SMQ040_bin + ALQ130 + ' + ' + '.join(RACE_DUMMIES))
mod_int = smf.glm(formula=formula_int, data=df,
                  family=sm.families.Binomial(),
                  freq_weights=df['WTPOOL']).fit(disp=0)

int_p = mod_int.pvalues.get('log_CRP:RIAGENDR', np.nan)
print(f'CRP x Sex interaction p-value: {int_p:.4f}')
print()

for sex_val, sex_label in [(1, 'Male'), (2, 'Female')]:
    sub = df[df['RIAGENDR'] == sex_val]
    cov_sex = ['RIDAGEYR', 'BMXBMI', 'SMQ040_bin', 'ALQ130'] + RACE_DUMMIES
    formula_s = 'depression ~ log_CRP + ' + ' + '.join(cov_sex)
    mod_s = smf.glm(formula=formula_s, data=sub,
                    family=sm.families.Binomial(),
                    freq_weights=sub['WTPOOL']).fit(disp=0)
    OR = np.exp(mod_s.params['log_CRP'])
    CI = np.exp(mod_s.conf_int().loc['log_CRP'])
    p  = mod_s.pvalues['log_CRP']
    print(f'{sex_label} (N={len(sub):,}): OR={OR:.3f} (95% CI {CI[0]:.3f}-{CI[1]:.3f}), p={p:.3f}')

## 7. Weighted Linear Regression: PHQ-9 Continuous Score

Coefficient (beta) = change in PHQ-9 total per 1-unit increase in the (log-transformed) marker.

In [ ]:
wls_rows = []
for marker in MARKERS:
    for model_type, extra in [('unadjusted', []), ('adjusted', COVARIATES)]:
        predictors = [marker] + extra
        X = sm.add_constant(df[predictors]).astype(float)
        mod = sm.WLS(df['PHQ9_total'], X, weights=df['WTPOOL']).fit()
        wls_rows.append({
            'marker':   marker,
            'model':    model_type,
            'beta':     mod.params[marker],
            'CI_lower': mod.conf_int().loc[marker, 0],
            'CI_upper': mod.conf_int().loc[marker, 1],
            'pvalue':   mod.pvalues[marker],
        })

wls_df = pd.DataFrame(wls_rows)
adj_mask = wls_df['model'] == 'adjusted'
_, pvals_fdr_wls, _, _ = multipletests(
    wls_df.loc[adj_mask, 'pvalue'].fillna(1), method='fdr_bh'
)
wls_df.loc[adj_mask, 'pvalue_fdr'] = pvals_fdr_wls
wls_df['marker_label'] = wls_df['marker'].map(MARKER_LABELS)

print('Adjusted WLS results (outcome: PHQ-9 total score):')
print(wls_df[adj_mask][['marker_label','beta','CI_lower','CI_upper','pvalue','pvalue_fdr']]
      .round(3).to_string(index=False))

wls_df.to_csv(PROCESSED / 'wls_phq9_results.csv', index=False)